# Lab 17: Feature-Based Machine Learning for Time Series

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-17-feature-based-machine-learning-lab.ipynb)

This lab is designed for independent study. You should be able to read the explanations, run the code, change the parameters, and answer the interpretation questions without needing a separate lecture.

## Main idea

Classical time-series models such as AR, MA, ARMA, ARIMA, and SARIMA describe temporal dependence through equations involving past observations and past noise terms. In a feature-based machine-learning approach, we convert a time series into a supervised-learning table.

Instead of fitting only a formula such as

$$
Y_t = \phi_1 Y_{t-1} + \cdots + \phi_p Y_{t-p} + W_t,
$$

we build a feature vector from information available at a forecast origin time $t$:

$$
z_t = igl(Y_t, Y_{t-1}, Y_{t-7}, 	ext{rolling means}, 	ext{calendar features}, 	ext{exogenous variables}igr),
$$

and train a model to predict a future value:

$$
Y_{t+h} pprox f(z_t).
$$

Here $h$ is the forecast horizon. For example, $h=1$ means one-step-ahead forecasting and $h=7$ means seven-step-ahead forecasting.

## Learning goals

By the end of this lab, you should be able to:

1. Turn a univariate or exogenous time series into a supervised-learning data set.
2. Build lag features, rolling-window features, calendar features, Fourier features, and exogenous features.
3. Use chronological train/test splitting instead of random splitting.
4. Compare machine-learning forecasts with naive and seasonal-naive baselines.
5. Identify and avoid common time-series leakage mistakes.
6. Use direct and recursive strategies for multi-step forecasting.
7. Interpret feature importance and residual diagnostics.

## Packages used

This lab uses standard Colab-friendly packages: `numpy`, `pandas`, `matplotlib`, and `scikit-learn`.


## 1. Mathematical background: supervised learning with a forecast origin

Let $Y_t$ be the time series we want to forecast. At time $t$, we are allowed to use only information already available at or before time $t$. We call this the information set

$$
I_t = \{Y_s: s \leq t\} \cup \{	ext{known covariates available at time } t\}.
$$

A feature vector $z_t$ is valid for forecasting $Y_{t+h}$ only if every component of $z_t$ is measurable with respect to $I_t$. In practical terms:

- $Y_t$, $Y_{t-1}$, $Y_{t-7}$ are valid lag features.
- A rolling mean of past observations, such as $rac{1}{7}\sum_{j=0}^6 Y_{t-j}$, is valid.
- A centered rolling mean using $Y_{t+1}$ or $Y_{t+2}$ is not valid.
- Calendar features such as day of week are valid because they are known in advance.
- Future weather variables are valid only if they are truly available as forecasts at the forecast origin.

A supervised-learning data set has rows

$$
(z_t, Y_{t+h}), \quad t = t_0, t_0+1, \ldots, T-h.
$$

The model is trained by minimizing a loss function such as mean squared error:

$$
\min_f \sum_t igl(Y_{t+h} - f(z_t)igr)^2.
$$

The most important rule is: do not let the features accidentally contain information from the target period.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(7339)
plt.rcParams["figure.figsize"] = (10, 4)


## 2. Create a realistic synthetic data set

We will simulate daily demand. The simulated series contains:

- a slow upward trend,
- weekly seasonality,
- annual seasonality,
- a temperature-like exogenous variable,
- autoregressive dependence,
- random noise.

The point of using synthetic data is that we know the data-generating structure. This makes it easier to understand which features should help.


In [ ]:
def simulate_daily_demand(n_days=4 * 365, seed=7339):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2020-01-01", periods=n_days, freq="D")
    t = np.arange(n_days)

    # Known calendar patterns.
    weekly = 8 * np.sin(2 * np.pi * t / 7)
    annual = 12 * np.sin(2 * np.pi * t / 365.25)
    trend = 0.015 * t

    # Temperature-like exogenous variable.
    temp = 60 + 18 * np.sin(2 * np.pi * (t - 80) / 365.25) + rng.normal(0, 4, n_days)

    # Demand depends on seasonality, trend, temperature, and its own previous value.
    demand = np.zeros(n_days)
    noise = rng.normal(0, 5, n_days)
    base = 120 + trend + weekly + annual + 0.55 * np.maximum(temp - 65, 0)

    demand[0] = base[0] + noise[0]
    for i in range(1, n_days):
        demand[i] = 0.55 * demand[i - 1] + 0.45 * base[i] + noise[i]

    return pd.DataFrame({"date": dates, "y": demand, "temp": temp})

raw = simulate_daily_demand()
raw.head()


In [ ]:
fig, ax = plt.subplots()
ax.plot(raw["date"], raw["y"], label="demand")
ax.set_title("Synthetic daily demand")
ax.set_xlabel("date")
ax.set_ylabel("demand")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(raw["date"], raw["temp"], label="temperature-like exogenous variable")
ax.set_title("Exogenous variable")
ax.set_xlabel("date")
ax.set_ylabel("temp")
ax.legend()
plt.show()


### Checkpoint 1

Before building any model, answer these questions:

1. Do you see trend?
2. Do you see seasonality?
3. Is the exogenous variable itself seasonal?
4. Why might a model using only yesterday's demand miss important information?


## 3. Baseline forecasts

A machine-learning model is not useful unless it improves on simple baselines. We will use two baselines:

1. **Naive forecast:**

$$
\hat Y_{t+1|t} = Y_t.
$$

2. **Seasonal naive forecast with weekly period 7:**

$$
\hat Y_{t+1|t} = Y_{t+1-7}.
$$

The seasonal naive forecast says: forecast tomorrow using the value from the same day last week.


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.abs(y_true) > 1e-12
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def summarize_errors(y_true, y_pred, name):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE_percent": mape(y_true, y_pred),
    }

# Use the last 180 days as a test period.
test_days = 180
train_raw = raw.iloc[:-test_days].copy()
test_raw = raw.iloc[-test_days:].copy()

# One-step-ahead baseline forecasts on the test period.
y_test = test_raw["y"].to_numpy()

# For each test date, the previous day and previous week are available in the full historical data.
full_y = raw["y"].to_numpy()
test_start = len(raw) - test_days
naive_pred = full_y[test_start - 1: len(raw) - 1]
seasonal_naive_pred = full_y[test_start - 7: len(raw) - 7]

baseline_results = pd.DataFrame([
    summarize_errors(y_test, naive_pred, "naive: yesterday"),
    summarize_errors(y_test, seasonal_naive_pred, "seasonal naive: last week"),
])
baseline_results


## 4. Feature engineering without leakage

The main function below creates a supervised-learning table. For a forecast horizon $h$, each row uses information available at time $t$ and predicts $Y_{t+h}$.

Important details:

- `y_lag_1` is $Y_t$.
- `y_lag_7` is $Y_{t-6}$ if the row index is aligned differently? To avoid confusion, we define features using `shift(lag - 1)` after choosing the row as forecast origin. However, a clearer convention is to create features at row time $t$ directly: `y_lag_1 = y_t`, `y_lag_7 = y_{t-6}`. In this notebook, the target is shifted upward by $h$, so the row date is the forecast origin.
- Rolling means use current and past values only.
- Calendar and Fourier features are known in advance.

For practical forecasting, you must write down the forecast origin explicitly before constructing features.


In [ ]:
def add_fourier_features(df, period, order, prefix):
    out = df.copy()
    t = np.arange(len(out))
    for k in range(1, order + 1):
        out[f"{prefix}_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        out[f"{prefix}_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return out


def make_supervised_features(df, horizon=1):
    out = df.copy()
    out = add_fourier_features(out, period=7, order=3, prefix="weekly")
    out = add_fourier_features(out, period=365.25, order=3, prefix="annual")

    out["dayofweek"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)

    # Lag features. Row t is the forecast origin.
    for lag in [1, 2, 3, 7, 14, 28]:
        out[f"y_lag_{lag}"] = out["y"].shift(lag - 1)

    # Rolling features available at the forecast origin.
    # These include y_t and past values, but not y_{t+1}.
    out["roll_mean_7"] = out["y"].rolling(7).mean()
    out["roll_mean_28"] = out["y"].rolling(28).mean()
    out["roll_std_28"] = out["y"].rolling(28).std()

    # Exogenous feature available at the forecast origin.
    out["temp_current"] = out["temp"]
    out["temp_hot"] = np.maximum(out["temp"] - 65, 0)
    out["temp_cold"] = np.maximum(55 - out["temp"], 0)

    # Target h days ahead.
    out["target"] = out["y"].shift(-horizon)

    feature_cols = [
        c for c in out.columns
        if c not in ["date", "y", "target"]
    ]

    supervised = out.dropna().reset_index(drop=True)
    return supervised, feature_cols

supervised, feature_cols = make_supervised_features(raw, horizon=1)
supervised[["date", "y", "target"] + feature_cols[:8]].head()


In [ ]:
print("Number of supervised rows:", len(supervised))
print("Number of features:", len(feature_cols))
print("First 15 features:")
for c in feature_cols[:15]:
    print("-", c)


### Checkpoint 2

Explain why the following features are valid or invalid for predicting tomorrow's demand:

1. yesterday's demand,
2. demand from the same day last week,
3. a centered 7-day rolling mean,
4. day of week,
5. tomorrow's true demand,
6. tomorrow's temperature forecast, if that forecast is available at the forecast origin.


## 5. Chronological train/test split

For independent observations, many machine-learning courses use random train/test splits. For time series, random splitting is usually inappropriate because it mixes future and past.

We split chronologically:

- train on the earlier part of the series,
- test on the later part of the series.


In [ ]:
cutoff_date = raw["date"].iloc[-test_days]
train_df = supervised[supervised["date"] < cutoff_date].copy()
test_df = supervised[supervised["date"] >= cutoff_date].copy()

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_test = test_df[feature_cols]
y_test_ml = test_df["target"]

print("Train date range:", train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Test date range:", test_df["date"].min().date(), "to", test_df["date"].max().date())
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 6. Fit feature-based ML models

We will compare three models:

1. **Ridge regression:** a regularized linear model.
2. **Random forest:** a nonlinear model that averages decision trees.
3. **Gradient boosting:** a nonlinear additive tree model.

The goal is not only to get a small test error. The goal is to understand which features help and whether the model beats baselines.


In [ ]:
models = {
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "random forest": RandomForestRegressor(
        n_estimators=120,
        max_depth=8,
        min_samples_leaf=5,
        random_state=7339,
        n_jobs=-1,
    ),
    "gradient boosting": GradientBoostingRegressor(
        n_estimators=180,
        learning_rate=0.05,
        max_depth=3,
        random_state=7339,
    ),
}

predictions = {}
rows = []

# Match baseline arrays to the supervised test period.
test_indices = test_df.index.to_numpy()
# Easier: compute baselines directly from row features.
naive_for_supervised_test = test_df["y_lag_1"].to_numpy()
seasonal_for_supervised_test = test_df["y_lag_7"].to_numpy()

rows.append(summarize_errors(y_test_ml, naive_for_supervised_test, "naive: y_t"))
rows.append(summarize_errors(y_test_ml, seasonal_for_supervised_test, "seasonal naive: y_{t-6}"))

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append(summarize_errors(y_test_ml, pred, name))

results = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
results


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_df["date"], y_test_ml, label="actual", linewidth=2)
ax.plot(test_df["date"], naive_for_supervised_test, label="naive", alpha=0.8)
ax.plot(test_df["date"], predictions["ridge"], label="ridge", alpha=0.9)
ax.plot(test_df["date"], predictions["gradient boosting"], label="gradient boosting", alpha=0.9)
ax.set_title("One-step-ahead forecasts on the test period")
ax.set_xlabel("date")
ax.set_ylabel("demand")
ax.legend()
plt.show()


### Checkpoint 3

1. Which model has the smallest MAE?
2. Does the best ML model beat the naive and seasonal-naive baselines?
3. Are the model forecasts too smooth, too noisy, or reasonably close to the observed values?
4. If a model does not beat the baseline, what might that mean?


## 7. Feature interpretation

Feature-based machine learning can be more interpretable than black-box sequence models if we inspect the features carefully. For tree models, we can look at impurity-based feature importance. For ridge regression, we can inspect standardized coefficients.

Feature importance is not causal importance. It tells us which variables are useful for prediction under the fitted model.


In [ ]:
rf = models["random forest"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Top 12 random-forest features:")
display(importances.head(12).to_frame("importance"))

fig, ax = plt.subplots(figsize=(8, 5))
importances.head(15).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top random-forest feature importances")
ax.set_xlabel("importance")
plt.show()


In [ ]:
ridge_pipeline = models["ridge"]
ridge_coef = ridge_pipeline.named_steps["ridge"].coef_
ridge_importance = pd.Series(ridge_coef, index=feature_cols).sort_values(key=lambda s: np.abs(s), ascending=False)

print("Top 12 ridge coefficients by absolute value:")
display(ridge_importance.head(12).to_frame("standardized coefficient"))


## 8. Residual diagnostics

For a forecast model, residuals on the test period are

$$
e_t = Y_{t+1} - \hat Y_{t+1|t}.
$$

Good residuals should have no obvious remaining pattern. If residuals still show trend, seasonality, or autocorrelation, the feature set may be missing information.


In [ ]:
best_name = results.iloc[0]["model"]
if best_name in predictions:
    best_pred = predictions[best_name]
else:
    best_pred = naive_for_supervised_test if "naive" in best_name else seasonal_for_supervised_test

resid = y_test_ml.to_numpy() - best_pred

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_df["date"], resid)
ax.axhline(0, linestyle="--")
ax.set_title(f"Test residuals for best model: {best_name}")
ax.set_xlabel("date")
ax.set_ylabel("residual")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(best_pred, resid, alpha=0.7)
ax.axhline(0, linestyle="--")
ax.set_title("Residuals versus fitted forecasts")
ax.set_xlabel("forecast")
ax.set_ylabel("residual")
plt.show()


In [ ]:
def sample_acf(x, max_lag=30):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.sum(x * x)
    vals = [1.0]
    for h in range(1, max_lag + 1):
        vals.append(np.sum(x[h:] * x[:-h]) / denom)
    return np.array(vals)

acf_vals = sample_acf(resid, max_lag=30)

fig, ax = plt.subplots(figsize=(9, 4))
ax.stem(np.arange(len(acf_vals)), acf_vals)
ax.axhline(0, linewidth=1)
ax.axhline(2 / np.sqrt(len(resid)), linestyle="--")
ax.axhline(-2 / np.sqrt(len(resid)), linestyle="--")
ax.set_title("Sample ACF of test residuals")
ax.set_xlabel("lag")
ax.set_ylabel("ACF")
plt.show()


### Checkpoint 4

1. Are residuals centered around zero?
2. Is there evidence of remaining weekly structure?
3. Is the residual ACF mostly inside the approximate bands?
4. Which new features would you add if residual autocorrelation remains?


## 9. A leakage demonstration

Leakage can make a model look excellent in a notebook but fail in the real world. A common leakage mistake is using a centered rolling feature. For a row with forecast origin $t$, a centered rolling mean may include values from $t+1$ or $t+2$, which are future values relative to the forecast origin.

We will create one intentionally invalid feature and compare the result. This section is meant as a warning.


In [ ]:
def make_leaky_features(df, horizon=1):
    supervised, feature_cols = make_supervised_features(df, horizon=horizon)

    # Intentionally invalid: centered rolling mean uses future observations.
    temp = df.copy()
    temp["leaky_centered_mean_7"] = temp["y"].rolling(7, center=True).mean()
    temp["target"] = temp["y"].shift(-horizon)
    temp = temp[["date", "leaky_centered_mean_7"]].dropna()

    merged = supervised.merge(temp, on="date", how="left").dropna().reset_index(drop=True)
    return merged, feature_cols + ["leaky_centered_mean_7"]

leaky_df, leaky_cols = make_leaky_features(raw, horizon=1)
leaky_train = leaky_df[leaky_df["date"] < cutoff_date].copy()
leaky_test = leaky_df[leaky_df["date"] >= cutoff_date].copy()

leaky_model = GradientBoostingRegressor(
    n_estimators=180,
    learning_rate=0.05,
    max_depth=3,
    random_state=7339,
)
leaky_model.fit(leaky_train[leaky_cols], leaky_train["target"])
leaky_pred = leaky_model.predict(leaky_test[leaky_cols])

leaky_comparison = pd.DataFrame([
    summarize_errors(y_test_ml.loc[leaky_test.index[-len(leaky_pred):]] if False else leaky_test["target"], leaky_pred, "INVALID leaky model"),
    summarize_errors(y_test_ml, predictions["gradient boosting"], "valid gradient boosting"),
])
leaky_comparison


The invalid model may look artificially strong because it has access to information that would not be known when making a real forecast. In a project report, always explain why every feature is available at the forecast origin.


## 10. Direct multi-step forecasting

For horizon $h > 1$, there are several strategies.

### Direct strategy

Train one separate model for each horizon:

$$
Y_{t+h} pprox f_h(z_t).
$$

This is simple and avoids feeding model predictions back into the input. The disadvantage is that we need to train multiple models.


In [ ]:
horizons = [1, 3, 7, 14]
rows = []
direct_predictions = {}

for h in horizons:
    sdf, cols = make_supervised_features(raw, horizon=h)
    train_h = sdf[sdf["date"] < cutoff_date].copy()
    test_h = sdf[sdf["date"] >= cutoff_date].copy()

    X_train_h = train_h[cols]
    y_train_h = train_h["target"]
    X_test_h = test_h[cols]
    y_test_h = test_h["target"]

    model_h = GradientBoostingRegressor(
        n_estimators=180,
        learning_rate=0.05,
        max_depth=3,
        random_state=7339,
    )
    model_h.fit(X_train_h, y_train_h)
    pred_h = model_h.predict(X_test_h)
    direct_predictions[h] = (test_h["date"].to_numpy(), y_test_h.to_numpy(), pred_h)

    rows.append({
        "horizon": h,
        "MAE": mean_absolute_error(y_test_h, pred_h),
        "RMSE": rmse(y_test_h, pred_h),
        "MAPE_percent": mape(y_test_h, pred_h),
    })

horizon_results = pd.DataFrame(rows)
horizon_results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(horizon_results["horizon"], horizon_results["MAE"], marker="o")
ax.set_title("Direct forecasting error by horizon")
ax.set_xlabel("forecast horizon h")
ax.set_ylabel("MAE")
plt.show()


### Checkpoint 5

1. Does error increase as the horizon increases?
2. Why should longer-horizon forecasts usually be harder?
3. Why might direct models for different horizons learn different feature relationships?


## 11. Recursive forecasting

The recursive strategy trains a one-step-ahead model and then repeatedly feeds its own predictions back into the feature construction process.

This is useful when we need a forecast path, but it has a risk: errors can accumulate.

Below is a simplified recursive forecast for the last 14 days. To keep the code readable, we use only lag and calendar/exogenous features needed by the already trained one-step feature model.


In [ ]:
def one_row_features(history_y, date, temp_value):
    # history_y contains all observed or predicted values up to the forecast origin.
    temp_df = pd.DataFrame({
        "date": [date],
        "temp": [temp_value],
    })
    t_index = (date - raw["date"].iloc[0]).days

    feats = {}
    feats["temp"] = temp_value
    feats["dayofweek"] = date.dayofweek
    feats["month"] = date.month
    feats["is_weekend"] = int(date.dayofweek >= 5)

    for k in range(1, 4):
        feats[f"weekly_sin_{k}"] = np.sin(2 * np.pi * k * t_index / 7)
        feats[f"weekly_cos_{k}"] = np.cos(2 * np.pi * k * t_index / 7)
        feats[f"annual_sin_{k}"] = np.sin(2 * np.pi * k * t_index / 365.25)
        feats[f"annual_cos_{k}"] = np.cos(2 * np.pi * k * t_index / 365.25)

    for lag in [1, 2, 3, 7, 14, 28]:
        feats[f"y_lag_{lag}"] = history_y[-lag]

    feats["roll_mean_7"] = np.mean(history_y[-7:])
    feats["roll_mean_28"] = np.mean(history_y[-28:])
    feats["roll_std_28"] = np.std(history_y[-28:], ddof=1)
    feats["temp_current"] = temp_value
    feats["temp_hot"] = max(temp_value - 65, 0)
    feats["temp_cold"] = max(55 - temp_value, 0)

    return pd.DataFrame([feats])[feature_cols]

# Train one-step gradient boosting on all data before the final 14 days.
forecast_horizon = 14
recursive_train_raw = raw.iloc[:-forecast_horizon].copy()
recursive_future = raw.iloc[-forecast_horizon:].copy()

sdf_one, cols_one = make_supervised_features(recursive_train_raw, horizon=1)
rec_model = GradientBoostingRegressor(
    n_estimators=180,
    learning_rate=0.05,
    max_depth=3,
    random_state=7339,
)
rec_model.fit(sdf_one[cols_one], sdf_one["target"])

history = list(recursive_train_raw["y"].to_numpy())
recursive_preds = []

for _, row in recursive_future.iterrows():
    x_row = one_row_features(history, row["date"], row["temp"])
    yhat = rec_model.predict(x_row)[0]
    recursive_preds.append(yhat)
    history.append(yhat)

actual_future = recursive_future["y"].to_numpy()

print("Recursive 14-day MAE:", mean_absolute_error(actual_future, recursive_preds))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(recursive_future["date"], actual_future, marker="o", label="actual")
ax.plot(recursive_future["date"], recursive_preds, marker="o", label="recursive forecast")
ax.set_title("Recursive 14-day forecast path")
ax.set_xlabel("date")
ax.set_ylabel("demand")
ax.legend()
plt.show()


## 12. Mini-project: build your own feature set

Choose one of the following tasks.

### Option A: Feature ablation

Remove one feature group at a time:

1. lag features,
2. rolling features,
3. calendar/Fourier features,
4. temperature features.

Which group matters most?

### Option B: Horizon comparison

Compare direct forecasts for $h=1, 3, 7, 14, 28$. Plot MAE versus horizon and explain the result.

### Option C: Leakage audit

Create a list of every feature in this lab. For each feature, write whether it is valid or invalid at forecast origin $t$.

### Option D: Model comparison

Compare ridge regression, random forest, and gradient boosting under the same chronological split. Which model is best? Which is easiest to explain?


## 13. AI-assisted study prompts

Use an AI assistant as a study partner, but verify its answer with code and mathematical reasoning.

### Prompt 1: Leakage review

> I am forecasting $Y_{t+1}$ from features built at time $t$. Here are my features: `y_lag_1`, `y_lag_7`, `roll_mean_7`, `centered_roll_mean_7`, `dayofweek`, and `temp_current`. Which features are valid and which may leak future information? Explain carefully.

### Prompt 2: Feature design

> I have daily data with weekly and yearly seasonality. Suggest valid lag, rolling, calendar, and Fourier features for one-step-ahead forecasting. For each feature, explain why it is available at the forecast origin.

### Prompt 3: Model diagnosis

> My gradient boosting time-series model has low training error but poor test error under chronological validation. Give a checklist of possible causes and how to test them.

### Prompt 4: Report writing

> Write a concise paragraph explaining why random train/test split is inappropriate for time-series forecasting and why chronological validation is used instead.


## 14. Lab checklist

Before leaving this lab, make sure you can do the following:

- [ ] Define a forecast origin and a forecast horizon.
- [ ] Construct lag features without using future values.
- [ ] Construct rolling features without using future values.
- [ ] Use calendar and Fourier features for seasonality.
- [ ] Build a chronological train/test split.
- [ ] Compare ML models with naive baselines.
- [ ] Identify at least one leakage mistake.
- [ ] Explain the difference between direct and recursive multi-step forecasting.
- [ ] Interpret feature importance cautiously.

## Key takeaway

Feature-based machine learning turns time-series forecasting into supervised learning, but it does not remove the temporal structure of the problem. The two most important habits are:

1. define the forecast origin clearly;
2. make sure every feature is available at that origin.
